In [ ]:
import os
import json
import requests
import openai

# File to store cached documentation
CACHE_FILE = "doc_cache.json"

def load_cache():
    """Load cached documentation from a JSON file."""
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r") as f:
            return json.load(f)
    return {}

def save_cache(cache):
    """Save the cache dictionary to a JSON file."""
    with open(CACHE_FILE, "w") as f:
        json.dump(cache, f)

def fetch_documentation(url, cache):
    """
    Fetch documentation content from a URL.
    If available in the cache, use that instead.
    """
    if url in cache:
        print("Using cached documentation.")
        return cache[url]
    else:
        print("Fetching documentation from URL.")
        response = requests.get(url)
        if response.status_code == 200:
            cache[url] = response.text
            save_cache(cache)
            return response.text
        else:
            raise Exception(f"Failed to fetch URL content. Status code: {response.status_code}")

def ask_question_using_cache(doc_text, question):
    """
    Ask ChatGPT a question using the cached documentation.
    The cached documentation (or an excerpt of it) is sent along with the question.
    """
    # Truncate if necessary to avoid token limits
    doc_excerpt = doc_text[:2000]
    
    messages = [
        {"role": "system", "content": "You are a helpful assistant that uses provided documentation as a reference."},
        {"role": "user", "content": f"Here is the cached documentation:\n\n{doc_excerpt}"},
        {"role": "user", "content": question}
    ]
    
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",  # Or "gpt-4" if available
        messages=messages,
        temperature=0.3
    )
    
    return response['choices'][0]['message']['content']

if __name__ == "__main__":
    # Set your OpenAI API key
    openai.api_key = "YOUR_OPENAI_API_KEY"
    
    # URL for the documentation (example: the README for the 'requests' library)
    doc_url = "https://raw.githubusercontent.com/psf/requests/main/README.md"
    
    # Load existing cache (or initialize an empty cache)
    cache = load_cache()
    
    # Fetch the documentation using the cache mechanism
    documentation = fetch_documentation(doc_url, cache)
    
    # Later, use the cached documentation to ask a question
    user_question = "Can you explain how to install and use this library based on the documentation provided?"
    answer = ask_question_using_cache(documentation, user_question)
    
    print("ChatGPT's Response:")
    print(answer)